In [1]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
BUCKET = "gs://retailsmart-lake-retailsmart-bd"

spark.conf.set("spark.sql.catalog.retailsmart", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.retailsmart.type", "hadoop")
spark.conf.set("spark.sql.catalog.retailsmart.warehouse", f"{BUCKET}/lakehouse")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailsmart.universal")

DataFrame[]

In [5]:
col = F.col

def leer(csv, **opciones):
    df = (spark.read.option("header", True).option("inferSchema", True)
          .options(**opciones).csv(f"{BUCKET}/landing/olist/{csv}"))
    for c in df.columns:
        if c.endswith("zip_code_prefix"):
            df = df.withColumn(c, F.lpad(F.col(c).cast("string"), 5, "0"))
    return df

def a_universal(csv, entidad, clave, reglas, dominio="logistica", **opciones):
    raw = leer(csv, **opciones)
    checked = raw.withColumn("motivo", F.coalesce(*[F.when(c, F.lit(t)) for t, c in reglas]))
    malos = checked.filter(col("motivo").isNotNull())
    buenos = checked.filter(col("motivo").isNull()).drop("motivo").dropDuplicates(clave)
    meta = lambda df: (df.withColumn("_fuente", F.lit("olist"))
                         .withColumn("_cargado_en", F.current_timestamp()))
    t_ok = f"retailsmart.universal.silver_{dominio}_{entidad}_v1"
    t_bad = f"retailsmart.universal.cuarentena_{dominio}_{entidad}_v1"
    meta(buenos).writeTo(t_ok).using("iceberg").createOrReplace()
    meta(malos).writeTo(t_bad).using("iceberg").createOrReplace()
    n_raw = raw.count()
    n_ok = spark.table(t_ok).count()
    n_bad = spark.table(t_bad).count()
    print(f"{entidad}: leídas {n_raw} | válidas {n_ok} | rechazadas {n_bad} | duplicados {n_raw - n_ok - n_bad}")

In [3]:
a_universal("olist_order_items_dataset.csv", "items_pedido", ["order_id", "order_item_id"], [
    ("order_id nulo", col("order_id").isNull()),
    ("product_id nulo", col("product_id").isNull()),
    ("seller_id nulo", col("seller_id").isNull()),
    ("precio negativo", col("price") < 0),
    ("flete negativo", col("freight_value") < 0)])

a_universal("olist_order_payments_dataset.csv", "pagos", ["order_id", "payment_sequential"], [
    ("order_id nulo", col("order_id").isNull()),
    ("monto negativo", col("payment_value") < 0),
    ("tipo de pago no definido", col("payment_type").isNull() | (col("payment_type") == "not_defined"))])

a_universal("olist_customers_dataset.csv", "clientes", ["customer_id"], [
    ("customer_id nulo", col("customer_id").isNull()),
    ("customer_unique_id nulo", col("customer_unique_id").isNull())])

a_universal("olist_sellers_dataset.csv", "vendedores", ["seller_id"], [
    ("seller_id nulo", col("seller_id").isNull())])

a_universal("olist_products_dataset.csv", "productos", ["product_id"], [
    ("product_id nulo", col("product_id").isNull()),
    ("peso negativo", col("product_weight_g") < 0)])

a_universal("product_category_name_translation.csv", "categorias", ["product_category_name"], [
    ("categoría nula", col("product_category_name").isNull())])

items_pedido: leídas 112650 | válidas 112650 | rechazadas 0 | duplicados 0


pagos: leídas 103886 | válidas 103883 | rechazadas 3 | duplicados 0


clientes: leídas 99441 | válidas 99441 | rechazadas 0 | duplicados 0


vendedores: leídas 3095 | válidas 3095 | rechazadas 0 | duplicados 0


productos: leídas 32951 | válidas 32951 | rechazadas 0 | duplicados 0


categorias: leídas 71 | válidas 71 | rechazadas 0 | duplicados 0


In [4]:
spark.table("retailsmart.universal.cuarentena_logistica_pagos_v1").show(truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+------------------------+-------+--------------------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|motivo                  |_fuente|_cargado_en               |
+--------------------------------+------------------+------------+--------------------+-------------+------------------------+-------+--------------------------+
|4637ca194b6387e2d538dc89b124b0ee|1                 |not_defined |1                   |0.0          |tipo de pago no definido|olist  |2026-10-18 02:40:43.474659|
|00b1cb0320190ca0daa2c88b35206009|1                 |not_defined |1                   |0.0          |tipo de pago no definido|olist  |2026-10-18 02:40:43.474659|
|c8c528189310eaa44a745b8d9d26908b|1                 |not_defined |1                   |0.0          |tipo de pago no definido|olist  |2026-10-18 02:40:43.474659|
+---------------------------

In [6]:
a_universal("olist_geolocation_dataset.csv", "geolocalizacion",
    ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
     "geolocation_city", "geolocation_state"], [
    ("código postal nulo", col("geolocation_zip_code_prefix").isNull()),
    ("coordenada nula", col("geolocation_lat").isNull() | col("geolocation_lng").isNull()),
    ("coordenada fuera de Brasil",
     ~(col("geolocation_lat").between(-34, 6) & col("geolocation_lng").between(-74, -34)))])

a_universal("olist_order_reviews_dataset.csv", "resenas", ["review_id", "order_id"], [
    ("review_id nulo", col("review_id").isNull()),
    ("order_id nulo", col("order_id").isNull()),
    ("puntaje fuera de 1 a 5",
     col("review_score").cast("int").isNull() | ~col("review_score").cast("int").between(1, 5))],
    multiLine=True, escape='"')

geolocalizacion: leídas 1000163 | válidas 738299 | rechazadas 42 | duplicados 261822


resenas: leídas 99224 | válidas 99224 | rechazadas 0 | duplicados 0


In [7]:
spark.table("retailsmart.universal.cuarentena_logistica_geolocalizacion_v1") \
     .groupBy("motivo").count().show(truncate=False)

+--------------------------+-----+
|motivo                    |count|
+--------------------------+-----+
|coordenada fuera de Brasil|42   |
+--------------------------+-----+

